In [53]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

# 1. Load Data
df = pd.read_csv('/content/archive (1).zip')

# Strip unwanted spaces from column names
df.columns = df.columns.str.strip()

# Drop index column if present
if 'Index' in df.columns:
    df = df.drop(columns=['Index'])

# 2. Match Feature Names Dynamically (fixes column name mismatch errors)
possible_features = [
    "Beats Per Minute (BPM)", "Energy", "Danceability",
    "Loudness (dB)", "Liveness", "Valence", "Acousticness", "Speechiness"
]

# Alternate short names if long names are missing
alt_names = {
    "Beats Per Minute (BPM)": "bpm",
    "Loudness (dB)": "loudness",
    "Energy": "energy",
    "Danceability": "danceability",
    "Liveness": "liveness",
    "Valence": "valence",
    "Acousticness": "acousticness",
    "Speechiness": "speechiness"
}

features = []
for f in possible_features:
    if f in df.columns:
        features.append(f)
    elif alt_names.get(f) in df.columns:
        features.append(alt_names[f])

if len(features) < 3:
    raise ValueError(f"Required features nahi mile! Available columns: {list(df.columns)}")

# 3. Handle Missing Values & Non-Numeric Types
# Features ko numeric me convert karein aur NaN rows hata dein
for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=features).reset_index(drop=True)
df2 = df[features]

# 4. MinMax Scaling
scaler = MinMaxScaler()
df2_scaled = scaler.fit_transform(df2)

# 5. K-Means Clustering
# n_clusters=10 generates labels 0 to 9
n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(df2_scaled)

# 6. Assign Cluster Labels cleanly
df["Music Segments"] = [f"Cluster {c + 1}" for c in clusters]

# 7. Dynamic Axis Selection for 3D Plot
# Jo actual columns present hain wahi plot me use honge
x_col = next((c for c in ["Beats Per Minute (BPM)", "bpm", features[0]] if c in df.columns), features[0])
y_col = next((c for c in ["Energy", "energy", features[1]] if c in df.columns), features[1])
z_col = next((c for c in ["Danceability", "danceability", features[2]] if c in df.columns), features[2])

# 8. 3D Plotly Visualization (Ordered Cluster 1 to 10)
PLOT = go.Figure()

# Proper numeric sorting: Cluster 1 to 10
ordered_clusters = [f"Cluster {i+1}" for i in range(n_clusters)]

for segment in ordered_clusters:
    segment_data = df[df["Music Segments"] == segment]
    if not segment_data.empty:
        PLOT.add_trace(go.Scatter3d(
            x=segment_data[x_col],
            y=segment_data[y_col],
            z=segment_data[z_col],
            mode='markers',
            marker=dict(size=4, opacity=0.8),
            name=segment
        ))

PLOT.update_traces(
    hovertemplate=f'<b>{x_col}:</b> %{{x}}<br><b>{y_col}:</b> %{{y}}<br><b>{z_col}:</b> %{{z}}<extra></extra>'
)

PLOT.update_layout(
    title="Spotify Music Clusters (3D Projection)",
    width=900,
    height=800,
    autosize=True,
    showlegend=True,
    scene=dict(
        xaxis=dict(title=x_col),
        yaxis=dict(title=y_col),
        zaxis=dict(title=z_col)
    ),
    font=dict(family="Arial, sans-serif", color='black', size=12)
)

PLOT.show()

In [54]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(df2_scaled, clusters, test_size=0.2, random_state=42)

# Classifier train karein
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Predictions aur Accuracy
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("--- Supervised Classification Accuracy ---")
print(f"Cluster Prediction Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

--- Supervised Classification Accuracy ---
Cluster Prediction Accuracy: 91.23%

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95        58
           1       0.88      0.95      0.91        44
           2       0.95      0.92      0.93        61
           3       0.88      0.92      0.90        48
           4       1.00      0.95      0.97        20
           5       0.82      0.90      0.86        31
           6       0.88      0.81      0.84        26
           7       0.93      0.95      0.94        40
           8       0.95      0.81      0.88        26
           9       0.89      0.91      0.90        45

    accuracy                           0.91       399
   macro avg       0.91      0.91      0.91       399
weighted avg       0.91      0.91      0.91       399

